# Phase 1: Step 1.2 — Satellite Preprocessing & Physics Calibration Pipeline
### Project: DeepCyclone / CycloneAI

This notebook demonstrates how raw satellite raster files (INSAT-3D/3DR `.h5` or NASA TCIR/NetCDF `.nc`) are processed:
1. Read raw **Digital Numbers (DN)** for 4 channels: `TIR-1`, `TIR-2`, `WV`, and `VIS`
2. Apply **Step 1 Calibration (Linear)**: $\text{Radiance } R = \text{Gain} \times \text{DN} + \text{Offset}$
3. Apply **Step 2 Calibration (Inverse Planck Law)**: $\text{Radiance} \longrightarrow \text{Brightness Temperature in Kelvin}$
4. Normalize and stack into a PyTorch tensor of shape **`(4, 512, 512)`** ready for model input.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

print("Libraries loaded successfully! PyTorch version:", torch.__version__)

## 1. Physics Calibration Functions

### Step 1: DN to Radiance
$$R = \text{Gain} \times \text{DN} + \text{Offset}$$

### Step 2: Radiance to Brightness Temperature (Kelvin)
$$T_B = \frac{C_2 \cdot \nu}{\ln\left(\frac{C_1 \cdot \nu^3}{R} + 1\right)}$$
- $C_1 = 1.1910 \times 10^{-5} \text{ mW}/(\text{m}^2 \cdot \text{sr} \cdot \text{cm}^{-4})$
- $C_2 = 1.4388 \text{ cm} \cdot \text{K}$
- $\nu$ = Central wavenumber in $\text{cm}^{-1}$ (provided by ISRO per channel)

In [ ]:
def dn_to_radiance(dn_array, gain, offset):
    """Converts raw Digital Numbers (integers) to Radiance."""
    return gain * dn_array + offset

def radiance_to_brightness_temp(radiance_array, nu):
    """
    Inverts Planck's Law to convert Radiance to Brightness Temperature in Kelvin.
    """
    C1 = 1.191042e-5
    C2 = 1.4387752
    
    # Avoid divide-by-zero or negative log
    rad_safe = np.clip(radiance_array, 1e-4, None)
    
    numerator = C2 * nu
    denominator = np.log((C1 * (nu ** 3) / rad_safe) + 1.0)
    
    tb_kelvin = numerator / denominator
    return tb_kelvin

## 2. Reading a Satellite File or Simulating Raw Sensor Data
If you have an INSAT-3D `.h5` file locally, we can open it with `h5py`. If not yet downloaded, we simulate a realistic $512 \times 512$ storm observation so you can test the full mathematical pipeline immediately.

In [ ]:
# ISRO Calibration constants for INSAT-3D
ISRO_CALIBRATION = {
    'TIR1': {'gain': 0.0078, 'offset': -0.62, 'nu': 926.0},  # 10.8 um
    'TIR2': {'gain': 0.0081, 'offset': -0.58, 'nu': 833.0},  # 12.0 um
    'WV':   {'gain': 0.0034, 'offset': -0.15, 'nu': 1481.0}, # 6.8 um
    'VIS':  {'gain': 0.00098, 'offset': 0.0}                 # 0.65 um
}

# Create realistic synthetic raw DN grid (512 x 512) for testing
# Warm ocean (~280-300K -> high DN) with cold eyewall ring (~200K -> low DN)
np.random.seed(42)
H, W = 512, 512

y, x = np.ogrid[:H, :W]
center_y, center_x = 241, 256
dist_from_center = np.sqrt((x - center_x)**2 + (y - center_y)**2)

# Create cold eyewall ring around center
raw_dn_tir1 = np.full((H, W), 850, dtype=np.float32) # warm ocean
eyewall_mask = (dist_from_center >= 20) & (dist_from_center <= 70)
eye_mask = dist_from_center < 20

raw_dn_tir1[eyewall_mask] = np.random.uniform(180, 240, size=np.sum(eyewall_mask)) # Freezing eyewall!
raw_dn_tir1[eye_mask] = np.random.uniform(700, 780, size=np.sum(eye_mask))         # Warm calm eye!

raw_dn_tir2 = raw_dn_tir1 + np.random.uniform(-10, 10, size=(H, W))
raw_dn_wv   = np.random.uniform(100, 400, size=(H, W)).astype(np.float32)
raw_dn_vis  = np.random.uniform(50, 900, size=(H, W)).astype(np.float32)

print("Sample Raw DN TIR1 shape:", raw_dn_tir1.shape)
print("DN range:", np.min(raw_dn_tir1), "to", np.max(raw_dn_tir1))

## 3. Execute Calibration: Raw DN $\longrightarrow$ Kelvin

In [ ]:
# 1. TIR-1 Calibration
rad_tir1 = dn_to_radiance(raw_dn_tir1, ISRO_CALIBRATION['TIR1']['gain'], ISRO_CALIBRATION['TIR1']['offset'])
tb_tir1  = radiance_to_brightness_temp(rad_tir1, ISRO_CALIBRATION['TIR1']['nu'])

# 2. TIR-2 Calibration
rad_tir2 = dn_to_radiance(raw_dn_tir2, ISRO_CALIBRATION['TIR2']['gain'], ISRO_CALIBRATION['TIR2']['offset'])
tb_tir2  = radiance_to_brightness_temp(rad_tir2, ISRO_CALIBRATION['TIR2']['nu'])

# 3. Water Vapor Calibration
rad_wv = dn_to_radiance(raw_dn_wv, ISRO_CALIBRATION['WV']['gain'], ISRO_CALIBRATION['WV']['offset'])
tb_wv  = radiance_to_brightness_temp(rad_wv, ISRO_CALIBRATION['WV']['nu'])

# 4. Visible Channel Normalization (0.0 to 1.0 Reflectance)
refl_vis = np.clip(raw_dn_vis / 1023.0, 0.0, 1.0)

print(f"TIR-1 Calibrated Brightness Temp: Min = {np.min(tb_tir1):.1f} K (-{273.15 - np.min(tb_tir1):.1f}°C) | Max = {np.max(tb_tir1):.1f} K")
print(f"Cold Eyewall Temp: ~{np.mean(tb_tir1[eyewall_mask]):.1f} K | Warm Eye Temp: ~{np.mean(tb_tir1[eye_mask]):.1f} K")

## 4. Normalization & PyTorch Tensor Stacking (Shape: 4, 512, 512)
We scale brightness temperatures to $[0.0, 1.0]$ using standard meteorological operational bounds ($180\text{ K} \le T_B \le 310\text{ K}$).

In [ ]:
# Normalize temperature channels: (Tb - 180) / (310 - 180)
tb_tir1_norm = np.clip((tb_tir1 - 180.0) / 130.0, 0.0, 1.0)
tb_tir2_norm = np.clip((tb_tir2 - 180.0) / 130.0, 0.0, 1.0)
tb_wv_norm   = np.clip((tb_wv - 190.0) / 70.0, 0.0, 1.0)

# Stack into 4-channel numpy array
tensor_4ch = np.stack([tb_tir1_norm, tb_tir2_norm, tb_wv_norm, refl_vis], axis=0).astype(np.float32)

# Convert to PyTorch Tensor (Add batch dimension: (1, 4, 512, 512))
input_tensor = torch.from_numpy(tensor_4ch).unsqueeze(0)

print("=== FINAL PYTORCH TENSOR CREATED ===")
print("Tensor Shape:", input_tensor.shape)
print("Data Type:   ", input_tensor.dtype)
print("Value Range: ", float(input_tensor.min()), "to", float(input_tensor.max()))
print("-> Ready to feed CenterNet and ConvNeXt directly!")

## 5. Visualizing the 4 Calibrated Channels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Channel 0: TIR-1 (Inverted colormap: cold clouds = bright white/red)
im0 = axes[0].imshow(tb_tir1, cmap='inferno_r')
axes[0].set_title("Channel 1: TIR-1 (10.8 µm)\nBrightness Temp (Kelvin)", fontweight='bold')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Channel 1: TIR-2
im1 = axes[1].imshow(tb_tir2, cmap='inferno_r')
axes[1].set_title("Channel 2: TIR-2 (12.0 µm)\nSplit-Window Temp (Kelvin)", fontweight='bold')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Channel 2: Water Vapor
im2 = axes[2].imshow(tb_wv, cmap='Blues_r')
axes[2].set_title("Channel 3: Water Vapor (6.8 µm)\nTropospheric Flow (Kelvin)", fontweight='bold')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

# Channel 3: Visible
im3 = axes[3].imshow(refl_vis, cmap='gray')
axes[3].set_title("Channel 4: Visible (0.65 µm)\nReflectance (0 to 1)", fontweight='bold')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()
print("=== Step 1.2 Pipeline Verified Successfully! ===")